In [6]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import LSTM, Dense, Input, GlobalAveragePooling2D, Attention, TimeDistributed
from tensorflow.keras.models import Model
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [7]:
data_dir = "D:/archive/train"

In [8]:
batch_size = 32
img_size = (224, 224)
num_classes = 2 

In [9]:
dataset = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    labels='inferred',
    label_mode='int',
    image_size=img_size,
    batch_size=batch_size,
    shuffle=True,
    validation_split=0.2,
    subset='both',
    seed=123
)

Found 93853 files belonging to 2 classes.
Using 75083 files for training.
Using 18770 files for validation.


In [10]:
train_dataset = dataset[0]
val_dataset = dataset[1]

In [11]:
test_dataset = val_dataset.take(int(0.5 * len(val_dataset)))
val_dataset = val_dataset.skip(int(0.5 * len(val_dataset)))

In [12]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.2),
])

In [ ]:
C:/Users/ASUS/Desktop/deepfake/path_to_your_model/combined_model.h5

In [13]:
train_dataset = train_dataset.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.experimental.AUTOTUNE
)

In [14]:
AUTOTUNE = tf.data.experimental.AUTOTUNE

In [15]:
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

In [16]:
def create_resnet_model(input_shape):
    base_model = ResNet50(include_top=False, input_shape=input_shape, weights='imagenet')
    x = GlobalAveragePooling2D()(base_model.output)
    model = Model(inputs=base_model.input, outputs=x)
    return model

resnet_input_shape = (224, 224, 3)
resnet_model = create_resnet_model(resnet_input_shape)

In [19]:
from tensorflow.keras.layers import Layer

In [20]:
class AttentionLSTM(Layer):
    def __init__(self, lstm_units=128):
        super(AttentionLSTM, self).__init__()
        self.lstm = LSTM(lstm_units, return_sequences=True)
        self.attention = Attention()

    def call(self, inputs):
        lstm_out = self.lstm(inputs)
        attention_out = self.attention([lstm_out, lstm_out])
        return tf.reduce_mean(attention_out, axis=1)

In [21]:

class ResNetFeatureExtractor(Layer):
    def __init__(self, resnet_model):
        super(ResNetFeatureExtractor, self).__init__()
        self.resnet_model = resnet_model

    def call(self, inputs):
        if len(inputs.shape) == 4:
            inputs = tf.expand_dims(inputs, axis=1)
        elif len(inputs.shape) != 5:
            raise ValueError("Expected input to be 4D or 5D, but got shape: {}".format(inputs.shape))

        batch_size, time_steps, height, width, channels = tf.shape(inputs)
        inputs = tf.reshape(inputs, (-1, height, width, channels))
        features = self.resnet_model(inputs)
        features = tf.reshape(features, (batch_size, time_steps, -1))
        return features

In [22]:
class CombinedModel(tf.keras.Model):
    def __init__(self, resnet_model, attention_lstm):
        super(CombinedModel, self).__init__()
        self.resnet_feature_extractor = ResNetFeatureExtractor(resnet_model)
        self.attention_lstm = attention_lstm

    def call(self, inputs):
        features = self.resnet_feature_extractor(inputs)
        lstm_features = self.attention_lstm(features)
        return lstm_features

In [23]:
attention_lstm_layer = AttentionLSTM()
combined_model = CombinedModel(resnet_model, attention_lstm_layer)

In [24]:
def extract_features(dataset, combined_model):
    features = []
    labels = []

    for images, label in dataset:
        try:
            print("Batch images shape:", images.shape)
            print("Batch labels shape:", label.shape)
            
            batch_features = combined_model(images, training=False)
            
            print("Batch features shape:", batch_features.shape)
            
            features.append(batch_features)
            labels.append(label)
        except Exception as e:
            print("Error:", e)
            break

    features = np.concatenate([f.numpy() for f in features], axis=0)
    labels = np.concatenate([l.numpy() for l in labels], axis=0)

    return features, labels

In [25]:
train_features, train_labels = extract_features(train_dataset, combined_model)
val_features, val_labels = extract_features(val_dataset, combined_model)
test_features, test_labels = extract_features(test_dataset, combined_model)

Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)


C:\Users\ASUS\anaconda3\lib\site-packages\keras\src\ops\nn.py:545: UserWarning: You are using a softmax over axis -1 of a tensor of shape (32, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch im

Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch im

Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch im

...

C:\Users\ASUS\anaconda3\lib\site-packages\keras\src\ops\nn.py:545: UserWarning: You are using a softmax over axis -1 of a tensor of shape (18, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


Batch features shape: (18, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch im

Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch im

Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch im

Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch images shape: (32, 224, 224, 3)
Batch labels shape: (32,)
Batch features shape: (32, 128)
Batch im

In [30]:
meta_learner = make_pipeline(StandardScaler(), LogisticRegression())
meta_learner.fit(train_features, train_labels)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression', LogisticRegression())])

In [27]:
val_predictions = meta_learner.predict(val_features)
val_accuracy = accuracy_score(val_labels, val_predictions)
print(f'Validation Accuracy: {val_accuracy:.2f}')

Validation Accuracy: 0.81


In [28]:
test_predictions = meta_learner.predict(test_features)
test_accuracy = accuracy_score(test_labels, test_predictions)
print(f'Test Accuracy: {test_accuracy:.2f}')

Test Accuracy: 0.81


In [29]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

# Assuming you have your test predictions and true labels
test_predictions = meta_learner.predict(test_features)

# Print classification report including precision, recall, and f1-score
report = classification_report(test_labels, test_predictions, target_names=['Class 0', 'Class 1'])
print(report)

# Compute individual metrics
precision = precision_score(test_labels, test_predictions, average='weighted')
recall = recall_score(test_labels, test_predictions, average='weighted')
f1 = f1_score(test_labels, test_predictions, average='weighted')

print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1 Score: {f1:.2f}')

              precision    recall  f1-score   support

     Class 0       0.83      0.96      0.89      7370
     Class 1       0.64      0.26      0.37      2006

    accuracy                           0.81      9376
   macro avg       0.73      0.61      0.63      9376
weighted avg       0.79      0.81      0.78      9376

Precision: 0.79
Recall: 0.81
F1 Score: 0.78
